In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import sklearn as skl
# import sktime
import random
import os
# from sktime.forecasting.model_selection import temporal_train_test_split
# from sktime.utils.plotting import plot_series

import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 50)

%matplotlib inline
from matplotlib import rc
plt.rc('font', family='NanumBarunGothic')
# plt.rcParams['font.family'] ='NanumBarunGothic'
plt.rcParams['axes.unicode_minus'] = False

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
seed_everything(493) # Seed 고정

# 불러오기
train = pd.read_csv('./train.csv')
sales = pd.read_csv('./sales.csv')
product_info = pd.read_csv('./product_info.csv')
brand_keyword_cnt = pd.read_csv('./brand_keyword_cnt.csv')
sample_submission = pd.read_csv('./sample_submission.csv')
price = pd.read_csv('./price.csv')

# **가격대**

In [ ]:
id_vars = ['브랜드']
brand_keyword_cnt_melted = pd.melt(brand_keyword_cnt, id_vars=id_vars, var_name='날짜', value_name='연관키워드_언급량')

# 날짜 컬럼을 datetime 타입으로 변환합니다.
brand_keyword_cnt_melted['날짜'] = pd.to_datetime(brand_keyword_cnt_melted['날짜'])

# 판매량을 날짜로 그룹화하고, 판매량을 합산하여 정리합니다.
brand_keyword_cnt_grouped = brand_keyword_cnt_melted.groupby(['브랜드', '날짜'])['연관키워드_언급량'].sum().reset_index()

In [ ]:
id_vars = ['ID', '제품', '대분류', '중분류', '소분류', '브랜드']
train_melted = pd.melt(train, id_vars=id_vars, var_name='날짜', value_name='판매량')
train_melted['날짜'] = pd.to_datetime(train_melted['날짜'])
train_grouped = train_melted.groupby('ID')

groups = []
for i in tqdm(range(15889+1)):
    get_train = train_grouped.get_group(i).reset_index()
    get_train.drop('index', axis=1, inplace=True)
    groups.append(get_train)
train_vertical = pd.concat(groups, ignore_index=True)

  0%|          | 0/15890 [00:00<?, ?it/s]

In [ ]:
# Function to fill the values as per the revised conditions
def fill_values(row):
    start_col = 6 # 7th column (in the actual data, this should be 6)
    fill_value = int(row[start_col]) # Convert to integer
    if fill_value == 0:
        # Find the first non-zero value after the starting column
        fill_value = int(next((value for value in row[start_col:] if value > 0), 0)) # Convert to integer
    for i in range(start_col, len(row)):
        if row[i] == 0:
            row[i] = fill_value
        elif row[i] != fill_value:
            fill_value = int(row[i]) # Convert to integer
    return row

# Apply the function to each row with tqdm
tqdm.pandas()
price_filled = price.progress_apply(fill_values, axis=1)

  0%|          | 0/15890 [00:00<?, ?it/s]

In [ ]:
id_vars = ['ID', '제품', '대분류', '중분류', '소분류', '브랜드']
price_melted = pd.melt(price_filled, id_vars=id_vars, var_name='날짜', value_name='price')
price_melted['날짜'] = pd.to_datetime(price_melted['날짜'])
price_grouped = price_melted.groupby('ID')

groups = []
for i in tqdm(range(15889+1)):
    get_price = price_grouped.get_group(i).reset_index()
    get_price.drop('index', axis=1, inplace=True)
    groups.append(get_price)
price_vertical = pd.concat(groups, ignore_index=True)

  0%|          | 0/15890 [00:00<?, ?it/s]

In [ ]:
# 2022년 1월 1일 기준 (유아식품 때문)
상반기_가격대 = price_vertical[price_vertical['날짜'] == '2022-01-01']

In [ ]:
# 0값은 바로 윗 행의 값으로 채우기
상반기_가격대['price'] = 상반기_가격대['price'].replace(0, method='ffill')

In [ ]:
# 가격(price) 값을 수정하는 함수 정의
def adjust_price(value):
    if value < 100:  # 십 단위
        if value % 10 >= 5:  # 일의자리 숫자가 5 이상인 경우
            value = (value // 10 + 1) * 10  # 올림 처리
        else:
            value = value // 10 * 10  # 내림 처리 (반올림)
    elif 100 <= value < 1000:  # 백 단위
        value = value // 100 * 100  # 십의자리까지 수를 0으로 처리
    elif 1000 <= value < 10000:  # 천 단위
        if (value // 100) % 10 in [8, 9]:  # 백의자리 숫자가 8 혹은 9인 경우
            value = (value // 1000 + 1) * 1000  # 올림 처리
        else:
            value = value // 100 * 100  # 십의자리까지 수를 0으로 처리
    elif 10000 <= value < 100000:  # 만 단위
        if (value // 1000) % 10 in [8, 9]:  # 천의자리 숫자가 8 혹은 9인 경우
            value = (value // 10000 + 1) * 10000  # 올림 처리
        else:
            value = value // 1000 * 1000  # 백의자리까지 수를 0으로 처리
    elif value >= 100000:  # 십만 단위 이상
        value = value // 10000 * 10000  # 천의자리까지 수를 0으로 처리
    return value


# 'price' 컬럼에 함수 적용
상반기_가격대['price'] = 상반기_가격대['price'].apply(adjust_price)

In [ ]:
# 가격 천,만,십만 단위만 남도록 수정
def adjust_price_2(value):
    # 천 단위의 숫자 조건
    if 1000 <= value < 10000:
        if (value // 100) % 10 in [8, 9]:  # 백의자리 숫자가 8 혹은 9인 경우
            value = (value // 1000 + 1) * 1000  # 올림 처리
        else:
            value = value // 1000 * 1000  # 백의자리까지 수를 0으로 처리

    # 만 단위의 숫자 조건
    elif 10000 <= value < 100000:
        if (value // 1000) % 10 in [8, 9]:  # 천의자리 숫자가 8 혹은 9인 경우
            value = (value // 10000 + 1) * 10000  # 올림 처리
        else:
            value = value // 10000 * 10000  # 천의자리까지 수를 0으로 처리

    # 십만 단위의 숫자 조건
    elif 100000 <= value < 1000000:
        if (value // 1000) % 10 in [8, 9]:  # 천의자리 숫자가 8 혹은 9인 경우
            value = (value // 100000 + 1) * 100000  # 올림 처리
        else:
           value = value // 100000 * 100000  # 천의자리까지 수를 0으로 처리

    return value


상반기_가격대['price'] = 상반기_가격대['price'].apply(adjust_price_2)

In [ ]:
상반기_가격대 = 상반기_가격대[['제품', '브랜드', 'price']].reset_index().drop('index', axis=1)

In [ ]:
상반기_가격대['price'].unique()

array([ 10000,  30000,   4000,   5000,   7000,  20000,  60000,   8000,
         3000,   2000,   1000,  50000,   6000,   9000,    500,  40000,
          700,  80000,  70000,    800,    600, 100000,  90000,    900,
          300,    100,    400, 300000, 200000, 600000, 500000, 400000,
          200,     20,     40,     70,     90,     50,     60,     80,
           30])

In [ ]:
상반기_가격대

,제품,브랜드,price
0,B002-00001-00001,B002-00001,10000
1,B002-00002-00001,B002-00002,30000
2,B002-00002-00002,B002-00002,10000
3,B002-00002-00003,B002-00002,4000
4,B002-00003-00001,B002-00003,5000
...,...,...,...
15885,B002-03799-00002,B002-03799,2000
15886,B002-03799-00003,B002-03799,20000
15887,B002-03799-00004,B002-03799,10000
15888,B002-03799-00005,B002-03799,10000


In [ ]:
상반기_가격대['price_rank'] = 상반기_가격대['price'].rank(ascending=False, method='dense')

In [ ]:
상반기_가격대['price_rank'] = 상반기_가격대['price_rank'].astype(int)

In [ ]:
상반기_가격대 = 상반기_가격대[['제품', 'price_rank']]

In [ ]:
# 빈 데이터프레임 생성
empty_data = train.copy()
empty_data.iloc[:, 6:] = 0

In [ ]:
empty_data['price_rank'] = 0

In [ ]:
col = empty_data.pop('price_rank')
empty_data.insert(6, 'price_rank', col)

In [ ]:
price_rank_map = 상반기_가격대.set_index('제품')['price_rank'].to_dict()
empty_data['price_rank'] = empty_data['제품'].map(price_rank_map).combine_first(empty_data['price_rank'])

In [ ]:
empty_data.drop(['ID', '제품', '대분류', '중분류', '소분류', '브랜드'], axis=1, inplace=True)

In [ ]:
empty_data

,price_rank,2022-01-01,2022-01-02,2022-01-03,2022-01-04,2022-01-05,2022-01-06,2022-01-07,2022-01-08,2022-01-09,2022-01-10,2022-01-11,2022-01-12,2022-01-13,2022-01-14,2022-01-15,2022-01-16,2022-01-17,2022-01-18,2022-01-19,2022-01-20,2022-01-21,2022-01-22,2022-01-23,2022-01-24,...,2023-03-11,2023-03-12,2023-03-13,2023-03-14,2023-03-15,2023-03-16,2023-03-17,2023-03-18,2023-03-19,2023-03-20,2023-03-21,2023-03-22,2023-03-23,2023-03-24,2023-03-25,2023-03-26,2023-03-27,2023-03-28,2023-03-29,2023-03-30,2023-03-31,2023-04-01,2023-04-02,2023-04-03,2023-04-04
0,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,13,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,21,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,20,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15885,23,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
15886,14,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
15887,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
15888,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
가격대_data = empty_data.copy()
for i in tqdm(range(len(가격대_data))):
  가격대_data.iloc[i, 1:] = 가격대_data.iloc[i, 0]
가격대_data.drop('price_rank', axis=1, inplace=True)

  0%|          | 0/15890 [00:00<?, ?it/s]

In [ ]:
가격대_data

,2022-01-01,2022-01-02,2022-01-03,2022-01-04,2022-01-05,2022-01-06,2022-01-07,2022-01-08,2022-01-09,2022-01-10,2022-01-11,2022-01-12,2022-01-13,2022-01-14,2022-01-15,2022-01-16,2022-01-17,2022-01-18,2022-01-19,2022-01-20,2022-01-21,2022-01-22,2022-01-23,2022-01-24,2022-01-25,...,2023-03-11,2023-03-12,2023-03-13,2023-03-14,2023-03-15,2023-03-16,2023-03-17,2023-03-18,2023-03-19,2023-03-20,2023-03-21,2023-03-22,2023-03-23,2023-03-24,2023-03-25,2023-03-26,2023-03-27,2023-03-28,2023-03-29,2023-03-30,2023-03-31,2023-04-01,2023-04-02,2023-04-03,2023-04-04
0,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,...,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15
1,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,...,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13
2,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,...,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15
3,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,...,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21
4,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,...,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15885,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,...,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23
15886,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,...,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14
15887,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,...,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15
15888,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,...,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15


In [ ]:
# 가격대_data.to_csv('./가격대_data.csv', index=False)

# **#############################################**

In [ ]:
train_vertical['price'] = 0
train_vertical['price'] = price_vertical['price']

In [ ]:
가격대 = train_vertical[train_vertical['날짜'] == '2022-01-01']
가격대 = 가격대[['제품', '소분류', '브랜드', 'price']]

In [ ]:
가격대 = 가격대.reset_index()
가격대.drop('index', axis=1, inplace=True)

In [ ]:
가격대[가격대['price']==0]

,제품,소분류,브랜드,price
143,B002-00066-00006,B002-C003-0003,B002-00066,0
240,B002-00072-00007,B002-C003-0023,B002-00072,0
407,B002-00107-00049,B002-C003-0025,B002-00107,0
513,B002-00126-00009,B002-C003-0023,B002-00126,0
696,B002-00172-00001,B002-C003-0011,B002-00172,0
...,...,...,...,...
14506,B002-03417-00117,B002-C003-0027,B002-03417,0
14736,B002-03487-00009,B002-C003-0025,B002-03487,0
14831,B002-03517-00003,B002-C003-0027,B002-03517,0
15501,B002-03706-00002,B002-C003-0002,B002-03706,0


In [ ]:
# 'price' 컬럼 값 처리 함수
import math
def process_price(price):
    if price % 100 == 0:
        return price
    elif price % 10000 == 0:
        return price // 10000 * 10000
    else:
        return math.ceil(price / 100) * 100

가격대['price'] = 가격대['price'].apply(process_price)

In [ ]:
가격대

# **#############################################**

In [ ]:
a = 가격대.groupby('브랜드')['소분류'].unique().reset_index(name='소분류')
a['보유하는 소분류 개수'] = a['소분류'].apply(lambda x: len(set(x)))

In [ ]:
a

,브랜드,소분류,보유하는 소분류 개수
0,B002-00001,[B002-C003-0038],1
1,B002-00002,[B002-C003-0044],1
2,B002-00003,[B002-C003-0003],1
3,B002-00005,"[B002-C003-0032, B002-C003-0034, B002-C003-003...",4
4,B002-00006,[B002-C003-0034],1
...,...,...,...
3165,B002-03794,[B002-C003-0033],1
3166,B002-03795,[B002-C003-0003],1
3167,B002-03796,"[B002-C003-0003, B002-C003-0004, B002-C003-0001]",3
3168,B002-03798,"[B002-C003-0021, B002-C003-0042, B002-C003-004...",4


In [ ]:
brand_keyword_cnt_grouped[brand_keyword_cnt_grouped['브랜드']=='B002-00238']

,브랜드,날짜,연관키워드_언급량
89964,B002-00238,2022-01-01,2.55294
89965,B002-00238,2022-01-02,2.14679
89966,B002-00238,2022-01-03,3.04612
89967,B002-00238,2022-01-04,4.20655
89968,B002-00238,2022-01-05,3.58282
...,...,...,...
90418,B002-00238,2023-03-31,5.71511
90419,B002-00238,2023-04-01,4.51116
90420,B002-00238,2023-04-02,4.23556
90421,B002-00238,2023-04-03,5.00435


In [ ]:
brand_keyword_cnt_grouped[brand_keyword_cnt_grouped['브랜드']=='B002-00244']

,브랜드,날짜,연관키워드_언급량
91341,B002-00244,2022-01-01,0.304520
91342,B002-00244,2022-01-02,0.522180
91343,B002-00244,2022-01-03,0.739694
91344,B002-00244,2022-01-04,0.942852
91345,B002-00244,2022-01-05,0.594684
...,...,...,...
91795,B002-00244,2023-03-31,0.971854
91796,B002-00244,2023-04-01,0.826845
91797,B002-00244,2023-04-02,0.899349
91798,B002-00244,2023-04-03,0.841346


In [ ]:
list_train = 가격대[가격대['브랜드'] == 'B002-00238']['제품'].unique().tolist()

for product in list_train:
    result = product_info[product_info['제품'] == product]
    if not result.empty:
        print(result.to_string(index=False))
    else:
        print(f"No information found for product: {product}")

              제품                                                                                                                                      제품특성
B002-00238-00001 생균:100억 CFU 1일 총 섭취량:1캡슐 제품용량:1개월분 제품타입:캡슐 섭취횟수:하루 한 번 섭취방법:물과 함께 섭취대상:성인남녀 주요 기능성(식약처인증):장건강 영양소 원료명(식약처고시):아연 :1개 100억 유산균 400mg x 30캡슐


# **#############################################**

In [ ]:
가격대.groupby(['소분류', '브랜드'])['price'].unique().reset_index(name='가격 리스트')

# **여기서부터 다시**

In [ ]:
가격대분류 = 가격대.groupby(['소분류', '브랜드'])['price'].unique().reset_index(name='가격 리스트')

In [ ]:
가격대분류['가격 리스트'] = 가격대분류['가격 리스트'].apply(lambda x: int(sum(x) / len(x)) if len(x) > 1 else x[0])

In [ ]:
가격대분류

In [ ]:
# 가격대분류.to_csv('./가격대분류.csv', index=False)

In [ ]:
# pd.read_csv('./가격대분류.csv')